**Name:** Aman Jha
**Roll Number:** 1024170215

## Q1: Build Your Personalized Knowledge Base

Take your college roll number, extract its digits, and build a pandas DataFrame with 6 FAQ entries — 4 fixed ones plus 2 built from the last two digits of the roll number.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

roll_number = "1024170215"
digits = [int(d) for d in roll_number]
print("Roll number:", roll_number)
print("Digits:", digits)

last_two = roll_number[-2:]
categories = ["billing", "account", "general"]

for d in last_two:
    d = int(d)
    print(f"digit {d} -> category[{d} % 3] = {categories[d % 3]}")

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

personal_entries = [
    {"question": "how do i update my registered mobile number",
     "answer": "Go to Profile > Contact Details and update your mobile number.",
     "keywords": "mobile number update contact", "category": "account"},
    {"question": "how do i contact customer support",
     "answer": "You can email support@college.edu or call our helpline.",
     "keywords": "contact support help", "category": "general"},
]

faq_entries = fixed_entries + personal_entries
df = pd.DataFrame(faq_entries)
df

Roll number: 1024170215
Digits: [1, 0, 2, 4, 1, 7, 0, 2, 1, 5]
digit 1 -> category[1 % 3] = account
digit 5 -> category[5 % 3] = general


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i update my registered mobile number,Go to Profile > Contact Details and update you...,mobile number update contact,account
5,how do i contact customer support,You can email support@college.edu or call our ...,contact support help,general


## Q2: Generate and Score a Hypothesis

A scoring function that takes a query string and returns all matching entries ranked by confidence. A keyword match counts more than a plain question-word match.

In [2]:
def score_entries(query, df):
    query_words = set(query.lower().split())
    results = []
    for _, row in df.iterrows():
        keyword_words = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())
        score = len(query_words & keyword_words) * 2 + len(query_words & question_words)
        if score > 0:
            results.append((row["question"], row["answer"], score))
    results.sort(key=lambda x: x[2], reverse=True)
    return results

for question, answer, score in score_entries("how can i pay my fee", df):
    print(score, "-", question, "-", answer)

9 - how can i pay the fee - You can pay via UPI, card, or net banking.
3 - what is the annual fee - The annual fee is Rs 500.
3 - how do i update my registered mobile number - Go to Profile > Contact Details and update your mobile number.
2 - how do i contact customer support - You can email support@college.edu or call our helpline.
1 - how to reset password - Go to Settings > Reset Password.


## Q3: Questions By Category

`same_category(category_name, df)` returns all questions belonging to a given category. Calling it with `"account"`, the category of one of the personalized entries from Q1.

In [3]:
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"].tolist()

print(same_category("account", df))

['how to reset password', 'how do i update my registered mobile number']


## Q4: Add a Keyword and Save to CSV

Pick one entry, add a new keyword typed by the user, and save the whole updated DataFrame to `1024170215_faq_data.csv`.

In [4]:
try:
    new_keyword = input("Enter a new keyword to add: ")
except EOFError:
    new_keyword = "helpline"

if not new_keyword:
    new_keyword = "helpline"

print("New keyword:", new_keyword)

target_question = "how do i contact customer support"
df.loc[df["question"] == target_question, "keywords"] = (
    df.loc[df["question"] == target_question, "keywords"] + " " + new_keyword
)

csv_filename = "1024170215_faq_data.csv"
df.to_csv(csv_filename, index=False)
print("Saved to", csv_filename)

df

New keyword: helpline
Saved to 1024170215_faq_data.csv


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i update my registered mobile number,Go to Profile > Contact Details and update you...,mobile number update contact,account
5,how do i contact customer support,You can email support@college.edu or call our ...,contact support help helpline,general


## Q5: FAQ Entries Per Category

Using `groupby` to count how many FAQ entries fall under each category.

In [5]:
category_counts = df.groupby("category").size()
print(category_counts)

category
account    2
billing    2
general    2
dtype: int64


## Q6: Handling Ties in the Scoring Function

The scoring function is updated so that when two or more entries tie for the highest score, all of them are printed instead of silently picking one. One query below produces a tie, and one does not.

In [6]:
def score_entries_v2(query, df):
    query_words = set(query.lower().split())
    results = []
    for _, row in df.iterrows():
        keyword_words = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())
        score = len(query_words & keyword_words) * 2 + len(query_words & question_words)
        if score > 0:
            results.append((row["question"], row["answer"], score))
    if not results:
        return results
    max_score = max(r[2] for r in results)
    top_matches = [r for r in results if r[2] == max_score]
    return top_matches

print("Query: 'fee' (this one ties)")
for question, answer, score in score_entries_v2("fee", df):
    print(score, "-", question, "-", answer)

print()
print("Query: 'reset password' (this one does not tie)")
for question, answer, score in score_entries_v2("reset password", df):
    print(score, "-", question, "-", answer)

Query: 'fee' (this one ties)
3 - what is the annual fee - The annual fee is Rs 500.
3 - how can i pay the fee - You can pay via UPI, card, or net banking.

Query: 'reset password' (this one does not tie)
6 - how to reset password - Go to Settings > Reset Password.
